# Student Training Academy Database Management System
### RDBMS Foundation, Schema Design, Keys and Basic SQL Using Python

This notebook implements a complete relational database system for a training academy using **SQLite** and **Python**, covering schema design, constraints, CRUD operations, SQL querying, constraint testing, schema inspection, and a bonus feedback module.

## Task 1: Create SQLite Database Using Python

We create a database file `training_academy.db` using Python's `sqlite3` library and enable foreign key enforcement (SQLite has this **off by default**).

In [42]:
import sqlite3
import pandas as pd
import os

DB_NAME = "training_academy.db"

# Start fresh each run so the notebook is fully reproducible
if os.path.exists(DB_NAME):
    os.remove(DB_NAME)

def connect_db(db_name):
    """Connect to SQLite DB and enable foreign key constraint enforcement."""
    conn = sqlite3.connect(db_name)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

conn = connect_db(DB_NAME)
print(f"Connected to '{DB_NAME}' with foreign key enforcement ON.")

Connected to 'training_academy.db' with foreign key enforcement ON.


## Reusable Python Functions

These three helper functions are used throughout the notebook for all database interactions.

In [43]:
def execute_query(conn, sql, params=None):
    """Execute an INSERT/UPDATE/DELETE/DDL statement and commit."""
    if params is None:
        params = []
    cursor = conn.execute(sql, params)
    conn.commit()
    return cursor

def read_query(conn, sql, params=None):
    """Run a SELECT query and return results as a Pandas DataFrame."""
    if params is None:
        params = []
    return pd.read_sql_query(sql, conn, params=params)

print("Reusable functions defined: connect_db(), execute_query(), read_query()")

Reusable functions defined: connect_db(), execute_query(), read_query()


## Task 2: Create Tables

We create all six tables with proper **primary keys**, **foreign keys**, `NOT NULL`, `UNIQUE`, `CHECK`, and `DEFAULT` constraints, including the composite unique constraint on `(student_id, course_id)` in `enrollments`.

**Relationships:**
- `departments` 1 → N `instructors`
- `departments` 1 → N `courses`
- `instructors` 1 → N `courses`
- `students` 1 → N `enrollments`
- `courses` 1 → N `enrollments`
- `enrollments` 1 → 1 `payments`

In [44]:
schema_sql = """
CREATE TABLE departments (
    department_id   INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL UNIQUE
);

CREATE TABLE students (
    student_id          INTEGER PRIMARY KEY,
    student_name        TEXT NOT NULL,
    email                TEXT NOT NULL UNIQUE,
    city                 TEXT,
    registration_date    DATE NOT NULL
);

CREATE TABLE instructors (
    instructor_id    INTEGER PRIMARY KEY,
    instructor_name  TEXT NOT NULL,
    email             TEXT NOT NULL UNIQUE,
    department_id     INTEGER NOT NULL,
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

CREATE TABLE courses (
    course_id       INTEGER PRIMARY KEY,
    course_name     TEXT NOT NULL,
    department_id    INTEGER NOT NULL,
    instructor_id    INTEGER NOT NULL,
    fee              REAL NOT NULL CHECK (fee >= 0),
    level            TEXT CHECK (level IN ('Beginner', 'Intermediate', 'Advanced')),
    FOREIGN KEY (department_id) REFERENCES departments(department_id),
    FOREIGN KEY (instructor_id) REFERENCES instructors(instructor_id)
);

CREATE TABLE enrollments (
    enrollment_id     INTEGER PRIMARY KEY,
    student_id         INTEGER NOT NULL,
    course_id          INTEGER NOT NULL,
    enrollment_date    DATE NOT NULL,
    status              TEXT DEFAULT 'Active' CHECK (status IN ('Active', 'Completed', 'Cancelled')),
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id),
    UNIQUE (student_id, course_id)
);

CREATE TABLE payments (
    payment_id       INTEGER PRIMARY KEY,
    enrollment_id     INTEGER NOT NULL UNIQUE,
    amount             REAL NOT NULL CHECK (amount >= 0),
    payment_date       DATE NOT NULL,
    payment_status     TEXT CHECK (payment_status IN ('Paid', 'Pending', 'Refunded')),
    FOREIGN KEY (enrollment_id) REFERENCES enrollments(enrollment_id)
);
"""

# executescript runs multiple statements separated by ;
conn.executescript(schema_sql)
conn.commit()
print("All 6 tables created successfully:")
print(read_query(conn, "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';"))

All 6 tables created successfully:
          name
0  departments
1     students
2  instructors
3      courses
4  enrollments
5     payments


## Task 3: Insert Sample Data

We insert sample records in the correct order so foreign key constraints don't fail:

**departments → students → instructors → courses → enrollments → payments**

In [45]:
# --- Departments ---
departments = [
    (1, "Data Science"),
    (2, "Software Engineering"),
    (3, "Business Analytics"),
]
execute_query(conn, "INSERT INTO departments VALUES (?, ?);", None) if False else None
for d in departments:
    execute_query(conn, "INSERT INTO departments (department_id, department_name) VALUES (?, ?);", d)

print("Departments inserted.")

Departments inserted.


In [46]:
# --- Students ---
students = [
    (1, "Rahul Kumar", "rahul.kumar@example.com", "Patna", "2026-01-05"),
    (2, "Priya Singh", "priya.singh@example.com", "Kolkata", "2026-01-06"),
    (3, "Amit Raj", "amit.raj@example.com", "Delhi", "2026-01-07"),
    (4, "Sneha Verma", "sneha.verma@example.com", "Patna", "2026-01-10"),
    (5, "Aditya Sharma", "aditya.sharma@example.com", "Mumbai", "2026-01-12"),
]
for s in students:
    execute_query(conn, """INSERT INTO students
        (student_id, student_name, email, city, registration_date)
        VALUES (?, ?, ?, ?, ?);""", s)

print("Students inserted.")

Students inserted.


In [47]:
# --- Instructors ---
instructors = [
    (1, "Dr. Meera Iyer", "meera.iyer@example.com", 1),
    (2, "Arjun Sen", "arjun.sen@example.com", 2),
    (3, "Kavita Rao", "kavita.rao@example.com", 3),
]
for i in instructors:
    execute_query(conn, """INSERT INTO instructors
        (instructor_id, instructor_name, email, department_id)
        VALUES (?, ?, ?, ?);""", i)

print("Instructors inserted.")

Instructors inserted.


In [48]:
# --- Courses ---
courses = [
    (101, "Python for Beginners", 2, 2, 4999, "Beginner"),
    (102, "SQL and RDBMS Masterclass", 1, 1, 6999, "Beginner"),
    (103, "Machine Learning Basics", 1, 1, 11999, "Intermediate"),
    (104, "Business Dashboarding", 3, 3, 8999, "Intermediate"),
    (105, "Advanced Data Engineering", 1, 1, 15999, "Advanced"),
]
for c in courses:
    execute_query(conn, """INSERT INTO courses
        (course_id, course_name, department_id, instructor_id, fee, level)
        VALUES (?, ?, ?, ?, ?, ?);""", c)

print("Courses inserted.")

Courses inserted.


In [49]:
# --- Enrollments ---
enrollments = [
    (1001, 1, 101, "2026-02-01", "Active"),
    (1002, 1, 102, "2026-02-03", "Completed"),
    (1003, 2, 102, "2026-02-04", "Active"),
    (1004, 3, 103, "2026-02-05", "Active"),
    (1005, 4, 104, "2026-02-07", "Cancelled"),
]
for e in enrollments:
    execute_query(conn, """INSERT INTO enrollments
        (enrollment_id, student_id, course_id, enrollment_date, status)
        VALUES (?, ?, ?, ?, ?);""", e)

print("Enrollments inserted.")

Enrollments inserted.


In [50]:
# --- Payments ---
payments = [
    (501, 1001, 4999, "2026-02-01", "Paid"),
    (502, 1002, 6999, "2026-02-03", "Paid"),
    (503, 1003, 6999, "2026-02-04", "Pending"),
    (504, 1004, 11999, "2026-02-05", "Paid"),
    (505, 1005, 0, "2026-02-07", "Refunded"),
]
for p in payments:
    execute_query(conn, """INSERT INTO payments
        (payment_id, enrollment_id, amount, payment_date, payment_status)
        VALUES (?, ?, ?, ?, ?);""", p)

print("Payments inserted.")
print("\nAll sample data inserted successfully (3 departments, 5 students, 3 instructors, 5 courses, 5 enrollments, 5 payments).")

Payments inserted.

All sample data inserted successfully (3 departments, 5 students, 3 instructors, 5 courses, 5 enrollments, 5 payments).


## Task 4: Display All Tables

Displaying the contents of all tables using Pandas DataFrames.

In [51]:
print("DEPARTMENTS")
display(read_query(conn, "SELECT * FROM departments;"))

DEPARTMENTS


,department_id,department_name
0,1,Data Science
1,2,Software Engineering
2,3,Business Analytics


In [52]:
print("STUDENTS")
display(read_query(conn, "SELECT * FROM students;"))

STUDENTS


,student_id,student_name,email,city,registration_date
0,1,Rahul Kumar,rahul.kumar@example.com,Patna,2026-01-05
1,2,Priya Singh,priya.singh@example.com,Kolkata,2026-01-06
2,3,Amit Raj,amit.raj@example.com,Delhi,2026-01-07
3,4,Sneha Verma,sneha.verma@example.com,Patna,2026-01-10
4,5,Aditya Sharma,aditya.sharma@example.com,Mumbai,2026-01-12


In [53]:
print("INSTRUCTORS")
display(read_query(conn, "SELECT * FROM instructors;"))

INSTRUCTORS


,instructor_id,instructor_name,email,department_id
0,1,Dr. Meera Iyer,meera.iyer@example.com,1
1,2,Arjun Sen,arjun.sen@example.com,2
2,3,Kavita Rao,kavita.rao@example.com,3


In [54]:
print("COURSES")
display(read_query(conn, "SELECT * FROM courses;"))

COURSES


,course_id,course_name,department_id,instructor_id,fee,level
0,101,Python for Beginners,2,2,4999.0,Beginner
1,102,SQL and RDBMS Masterclass,1,1,6999.0,Beginner
2,103,Machine Learning Basics,1,1,11999.0,Intermediate
3,104,Business Dashboarding,3,3,8999.0,Intermediate
4,105,Advanced Data Engineering,1,1,15999.0,Advanced


In [55]:
print("ENROLLMENTS")
display(read_query(conn, "SELECT * FROM enrollments;"))

ENROLLMENTS


,enrollment_id,student_id,course_id,enrollment_date,status
0,1001,1,101,2026-02-01,Active
1,1002,1,102,2026-02-03,Completed
2,1003,2,102,2026-02-04,Active
3,1004,3,103,2026-02-05,Active
4,1005,4,104,2026-02-07,Cancelled


In [56]:
print("PAYMENTS")
display(read_query(conn, "SELECT * FROM payments;"))

PAYMENTS


,payment_id,enrollment_id,amount,payment_date,payment_status
0,501,1001,4999.0,2026-02-01,Paid
1,502,1002,6999.0,2026-02-03,Paid
2,503,1003,6999.0,2026-02-04,Pending
3,504,1004,11999.0,2026-02-05,Paid
4,505,1005,0.0,2026-02-07,Refunded


## Task 5: Basic SQL Queries

Running each of the required SQL queries and displaying the results.

In [57]:
# 1. Show all students
display(read_query(conn, "SELECT * FROM students;"))

,student_id,student_name,email,city,registration_date
0,1,Rahul Kumar,rahul.kumar@example.com,Patna,2026-01-05
1,2,Priya Singh,priya.singh@example.com,Kolkata,2026-01-06
2,3,Amit Raj,amit.raj@example.com,Delhi,2026-01-07
3,4,Sneha Verma,sneha.verma@example.com,Patna,2026-01-10
4,5,Aditya Sharma,aditya.sharma@example.com,Mumbai,2026-01-12


In [58]:
# 2. Show only student name, email, and city
display(read_query(conn, "SELECT student_name, email, city FROM students;"))

,student_name,email,city
0,Rahul Kumar,rahul.kumar@example.com,Patna
1,Priya Singh,priya.singh@example.com,Kolkata
2,Amit Raj,amit.raj@example.com,Delhi
3,Sneha Verma,sneha.verma@example.com,Patna
4,Aditya Sharma,aditya.sharma@example.com,Mumbai


In [59]:
# 3. Show all students from Patna
display(read_query(conn, "SELECT * FROM students WHERE city = 'Patna';"))

,student_id,student_name,email,city,registration_date
0,1,Rahul Kumar,rahul.kumar@example.com,Patna,2026-01-05
1,4,Sneha Verma,sneha.verma@example.com,Patna,2026-01-10


In [60]:
# 4. Show all courses with fee greater than 7000
display(read_query(conn, "SELECT * FROM courses WHERE fee > 7000;"))

,course_id,course_name,department_id,instructor_id,fee,level
0,103,Machine Learning Basics,1,1,11999.0,Intermediate
1,104,Business Dashboarding,3,3,8999.0,Intermediate
2,105,Advanced Data Engineering,1,1,15999.0,Advanced


In [61]:
# 5. Show all beginner-level courses
display(read_query(conn, "SELECT * FROM courses WHERE level = 'Beginner';"))

,course_id,course_name,department_id,instructor_id,fee,level
0,101,Python for Beginners,2,2,4999.0,Beginner
1,102,SQL and RDBMS Masterclass,1,1,6999.0,Beginner


In [62]:
# 6. Show all students sorted by student name
display(read_query(conn, "SELECT * FROM students ORDER BY student_name ASC;"))

,student_id,student_name,email,city,registration_date
0,5,Aditya Sharma,aditya.sharma@example.com,Mumbai,2026-01-12
1,3,Amit Raj,amit.raj@example.com,Delhi,2026-01-07
2,2,Priya Singh,priya.singh@example.com,Kolkata,2026-01-06
3,1,Rahul Kumar,rahul.kumar@example.com,Patna,2026-01-05
4,4,Sneha Verma,sneha.verma@example.com,Patna,2026-01-10


In [63]:
# 7. Show the top 3 highest fee courses
display(read_query(conn, "SELECT course_name, fee FROM courses ORDER BY fee DESC LIMIT 3;"))

,course_name,fee
0,Advanced Data Engineering,15999.0
1,Machine Learning Basics,11999.0
2,Business Dashboarding,8999.0


In [64]:
# 8. Show distinct student cities
display(read_query(conn, "SELECT DISTINCT city FROM students;"))

,city
0,Patna
1,Kolkata
2,Delhi
3,Mumbai


In [65]:
# 9. Show all pending payments
display(read_query(conn, "SELECT * FROM payments WHERE payment_status = 'Pending';"))

,payment_id,enrollment_id,amount,payment_date,payment_status
0,503,1003,6999.0,2026-02-04,Pending


In [66]:
# 10. Show all active enrollments
display(read_query(conn, "SELECT * FROM enrollments WHERE status = 'Active';"))

,enrollment_id,student_id,course_id,enrollment_date,status
0,1001,1,101,2026-02-01,Active
1,1003,2,102,2026-02-04,Active
2,1004,3,103,2026-02-05,Active


## Task 6: INSERT Operation Using Python

Adding a new student record using a parameterized query (prevents SQL injection and handles data types safely).

In [67]:
new_student = (6, "Rohan Das", "rohan.das@example.com", "Pune", "2026-02-15")

execute_query(conn, """INSERT INTO students
    (student_id, student_name, email, city, registration_date)
    VALUES (?, ?, ?, ?, ?);""", new_student)

print("New student inserted. Record:")
display(read_query(conn, "SELECT * FROM students WHERE student_id = ?;", (6,)))

New student inserted. Record:


,student_id,student_name,email,city,registration_date
0,6,Rohan Das,rohan.das@example.com,Pune,2026-02-15


## Task 7: UPDATE Operation Using Python

Updating the city of Rohan Das from Pune to Bengaluru.

In [68]:
execute_query(conn, "UPDATE students SET city = ? WHERE student_name = ?;", ("Bengaluru", "Rohan Das"))

print("Student record updated. New record:")
display(read_query(conn, "SELECT * FROM students WHERE student_name = ?;", ("Rohan Das",)))

Student record updated. New record:


,student_id,student_name,email,city,registration_date
0,6,Rohan Das,rohan.das@example.com,Bengaluru,2026-02-15


## Task 8: DELETE Operation Using Python

Deleting the student Rohan Das and confirming removal.

In [69]:
execute_query(conn, "DELETE FROM students WHERE student_name = ?;", ("Rohan Das",))

print("Student deleted. Remaining students:")
remaining = read_query(conn, "SELECT * FROM students;")
display(remaining)

still_present = "Rohan Das" in remaining["student_name"].values
print(f"\nIs 'Rohan Das' still present? {still_present}  --> Deletion {'FAILED' if still_present else 'confirmed successful'}.")

Student deleted. Remaining students:


,student_id,student_name,email,city,registration_date
0,1,Rahul Kumar,rahul.kumar@example.com,Patna,2026-01-05
1,2,Priya Singh,priya.singh@example.com,Kolkata,2026-01-06
2,3,Amit Raj,amit.raj@example.com,Delhi,2026-01-07
3,4,Sneha Verma,sneha.verma@example.com,Patna,2026-01-10
4,5,Aditya Sharma,aditya.sharma@example.com,Mumbai,2026-01-12



Is 'Rohan Das' still present? False  --> Deletion confirmed successful.


## Task 9: Constraint Testing

We intentionally attempt invalid operations to verify that the database correctly **rejects** them. Each test is wrapped in a `try/except` block to catch `sqlite3.IntegrityError`.

In [70]:
# Test 1: UNIQUE constraint -- insert a student with a duplicate email
print("TEST 1: UNIQUE constraint on email")
try:
    execute_query(conn, """INSERT INTO students
        (student_id, student_name, email, city, registration_date)
        VALUES (?, ?, ?, ?, ?);""", (99, "Duplicate Student", "rahul.kumar@example.com", "Patna", "2026-03-01"))
    print("ERROR: Insert succeeded but should have been rejected!")
except sqlite3.IntegrityError as e:
    print(f"PASSED -- Insert correctly rejected.\nReason: {e}")

TEST 1: UNIQUE constraint on email
PASSED -- Insert correctly rejected.
Reason: UNIQUE constraint failed: students.email


In [71]:
# Test 2: FOREIGN KEY constraint -- insert an enrollment with a non-existent student_id
print("TEST 2: FOREIGN KEY constraint on enrollments.student_id")
try:
    execute_query(conn, """INSERT INTO enrollments
        (enrollment_id, student_id, course_id, enrollment_date, status)
        VALUES (?, ?, ?, ?, ?);""", (9999, 9999, 101, "2026-03-01", "Active"))
    print("ERROR: Insert succeeded but should have been rejected!")
except sqlite3.IntegrityError as e:
    print(f"PASSED -- Insert correctly rejected.\nReason: {e}")

TEST 2: FOREIGN KEY constraint on enrollments.student_id
PASSED -- Insert correctly rejected.
Reason: FOREIGN KEY constraint failed


In [72]:
# Test 3: CHECK constraint -- insert a course with a negative fee
print("TEST 3: CHECK constraint on courses.fee (fee >= 0)")
try:
    execute_query(conn, """INSERT INTO courses
        (course_id, course_name, department_id, instructor_id, fee, level)
        VALUES (?, ?, ?, ?, ?, ?);""", (199, "Negative Fee Course", 1, 1, -500, "Beginner"))
    print("ERROR: Insert succeeded but should have been rejected!")
except sqlite3.IntegrityError as e:
    print(f"PASSED -- Insert correctly rejected.\nReason: {e}")

TEST 3: CHECK constraint on courses.fee (fee >= 0)
PASSED -- Insert correctly rejected.
Reason: CHECK constraint failed: fee >= 0


In [73]:
# Test 4: CHECK constraint on enrollments.status -- only Active, Completed, Cancelled allowed
print("TEST 4: CHECK constraint on enrollments.status")
try:
    execute_query(conn, """INSERT INTO enrollments
        (enrollment_id, student_id, course_id, enrollment_date, status)
        VALUES (?, ?, ?, ?, ?);""", (9998, 2, 103, "2026-03-01", "Started"))
    print("ERROR: Insert succeeded but should have been rejected!")
except sqlite3.IntegrityError as e:
    print(f"PASSED -- Insert correctly rejected.\nReason: {e}")

TEST 4: CHECK constraint on enrollments.status
PASSED -- Insert correctly rejected.
Reason: CHECK constraint failed: status IN ('Active', 'Completed', 'Cancelled')


In [74]:
print("All constraint tests completed -- the schema correctly enforces UNIQUE, FOREIGN KEY, and CHECK rules.")

All constraint tests completed -- the schema correctly enforces UNIQUE, FOREIGN KEY, and CHECK rules.


## Task 10: Schema Inspection

Inspecting table structures and foreign key relationships using SQLite's `PRAGMA` commands.

In [75]:
for table in ["students", "courses", "enrollments", "payments"]:
    print(f"--- Schema: {table} ---")
    display(read_query(conn, f"PRAGMA table_info({table});"))
    print()

--- Schema: students ---


,cid,name,type,notnull,dflt_value,pk
0,0,student_id,INTEGER,0,None,1
1,1,student_name,TEXT,1,None,0
2,2,email,TEXT,1,None,0
3,3,city,TEXT,0,None,0
4,4,registration_date,DATE,1,None,0



--- Schema: courses ---


,cid,name,type,notnull,dflt_value,pk
0,0,course_id,INTEGER,0,None,1
1,1,course_name,TEXT,1,None,0
2,2,department_id,INTEGER,1,None,0
3,3,instructor_id,INTEGER,1,None,0
4,4,fee,REAL,1,None,0
5,5,level,TEXT,0,None,0



--- Schema: enrollments ---


,cid,name,type,notnull,dflt_value,pk
0,0,enrollment_id,INTEGER,0,None,1
1,1,student_id,INTEGER,1,None,0
2,2,course_id,INTEGER,1,None,0
3,3,enrollment_date,DATE,1,None,0
4,4,status,TEXT,0,'Active',0



--- Schema: payments ---


,cid,name,type,notnull,dflt_value,pk
0,0,payment_id,INTEGER,0,None,1
1,1,enrollment_id,INTEGER,1,None,0
2,2,amount,REAL,1,None,0
3,3,payment_date,DATE,1,None,0
4,4,payment_status,TEXT,0,None,0


In [76]:
for table in ["students", "courses", "enrollments", "payments"]:
    print(f"--- Foreign Keys: {table} ---")
    fk_info = read_query(conn, f"PRAGMA foreign_key_list({table});")
    if fk_info.empty:
        print("(No foreign keys defined on this table)")
    else:
        display(fk_info)
    print()

--- Foreign Keys: students ---
(No foreign keys defined on this table)

--- Foreign Keys: courses ---


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,instructors,instructor_id,instructor_id,NO ACTION,NO ACTION,NONE
1,1,0,departments,department_id,department_id,NO ACTION,NO ACTION,NONE



--- Foreign Keys: enrollments ---


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,courses,course_id,course_id,NO ACTION,NO ACTION,NONE
1,1,0,students,student_id,student_id,NO ACTION,NO ACTION,NONE



--- Foreign Keys: payments ---


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,enrollments,enrollment_id,enrollment_id,NO ACTION,NO ACTION,NONE


## Bonus Task: course_feedback Table

Adding a new table to capture student feedback/ratings for completed courses, linked to `enrollments`.

In [77]:
execute_query(conn, """
CREATE TABLE course_feedback (
    feedback_id      INTEGER PRIMARY KEY,
    enrollment_id     INTEGER NOT NULL,
    rating             INTEGER CHECK (rating BETWEEN 1 AND 5),
    comments           TEXT,
    feedback_date      DATE NOT NULL,
    FOREIGN KEY (enrollment_id) REFERENCES enrollments(enrollment_id)
);
""")
print("Table 'course_feedback' created.")

Table 'course_feedback' created.


In [78]:
# Insert 3 feedback records
feedback_records = [
    (1, 1001, 5, "Excellent introduction to Python, very clear examples.", "2026-02-20"),
    (2, 1002, 4, "Good SQL coverage, could use more practice exercises.", "2026-02-22"),
    (3, 1004, 4, "Solid ML fundamentals course.", "2026-02-25"),
]
for f in feedback_records:
    execute_query(conn, """INSERT INTO course_feedback
        (feedback_id, enrollment_id, rating, comments, feedback_date)
        VALUES (?, ?, ?, ?, ?);""", f)

print("3 feedback records inserted.")

3 feedback records inserted.


In [79]:
# Show all feedback
print("All feedback:")
display(read_query(conn, "SELECT * FROM course_feedback;"))

All feedback:


,feedback_id,enrollment_id,rating,comments,feedback_date
0,1,1001,5,"Excellent introduction to Python, very clear e...",2026-02-20
1,2,1002,4,"Good SQL coverage, could use more practice exe...",2026-02-22
2,3,1004,4,Solid ML fundamentals course.,2026-02-25


In [80]:
# Show feedback with rating >= 4
print("Feedback with rating >= 4:")
display(read_query(conn, "SELECT * FROM course_feedback WHERE rating >= 4;"))

Feedback with rating >= 4:


,feedback_id,enrollment_id,rating,comments,feedback_date
0,1,1001,5,"Excellent introduction to Python, very clear e...",2026-02-20
1,2,1002,4,"Good SQL coverage, could use more practice exe...",2026-02-22
2,3,1004,4,Solid ML fundamentals course.,2026-02-25


In [81]:
# Try inserting rating as 6 -- should be rejected by CHECK constraint
print("Testing CHECK constraint: rating BETWEEN 1 AND 5 (attempting rating = 6)")
try:
    execute_query(conn, """INSERT INTO course_feedback
        (feedback_id, enrollment_id, rating, comments, feedback_date)
        VALUES (?, ?, ?, ?, ?);""", (4, 1003, 6, "Invalid rating test", "2026-02-26"))
    print("ERROR: Insert succeeded but should have been rejected!")
except sqlite3.IntegrityError as e:
    print(f"PASSED -- Insert correctly rejected.\nReason: {e}")

Testing CHECK constraint: rating BETWEEN 1 AND 5 (attempting rating = 6)
PASSED -- Insert correctly rejected.
Reason: CHECK constraint failed: rating BETWEEN 1 AND 5


## Mini Viva / Concept Questions

**1. What is an RDBMS?**
A Relational Database Management System is software that stores data in structured tables made up of rows and columns, where relationships between tables are defined using keys. Examples: SQLite, MySQL, PostgreSQL.

**2. What is the difference between a table and a schema?**
A table is a single structure that stores rows of related data (e.g., `students`). A schema is the overall blueprint describing all tables, their columns, data types, constraints, and relationships in the database.

**3. What is a primary key?**
A column (or combination of columns) that uniquely identifies each row in a table. It cannot be NULL and must be unique, e.g., `student_id` in `students`.

**4. What is a foreign key?**
A column in one table that references the primary key of another table, used to enforce a relationship between the two tables, e.g., `instructors.department_id` references `departments.department_id`.

**5. Why do we use constraints?**
Constraints (NOT NULL, UNIQUE, CHECK, DEFAULT, FOREIGN KEY) enforce data integrity rules at the database level, preventing invalid, duplicate, or inconsistent data from being stored, regardless of which application inserts the data.

**6. Why should email be marked as UNIQUE?**
Each student/instructor should be uniquely identifiable by their email, and duplicate emails would cause ambiguity in communication, login systems, and record identification.

**7. Why is UNIQUE(student_id, course_id) used in the enrollments table?**
This composite constraint ensures a student cannot enroll in the same course more than once, directly enforcing the business rule stated in the scenario.

**8. Why should foreign keys be enabled in SQLite?**
SQLite has foreign key enforcement **disabled by default** for backward compatibility. Without `PRAGMA foreign_keys = ON;`, invalid references (e.g., enrolling a non-existent student) would silently succeed, breaking referential integrity.

**9. What is the purpose of WHERE?**
The `WHERE` clause filters rows based on a condition, returning only the records that satisfy that condition (used in SELECT, UPDATE, and DELETE statements).

**10. What is the difference between UPDATE and DELETE?**
`UPDATE` modifies the values of existing columns in matching rows without removing the row. `DELETE` removes the entire matching row(s) from the table permanently.

**11. Why are parameterized queries safer?**
Parameterized queries (using `?` placeholders) separate SQL code from user-supplied data, preventing SQL injection attacks and correctly handling special characters, quotes, and data types automatically.

**12. Why should parent table records be inserted before child table records?**
Because child tables reference parent table primary keys via foreign keys; if the parent row doesn't exist yet, the foreign key constraint would reject the child row's insert (referential integrity requires the parent to exist first).

## Schema and Relationship Summary

| Table | Primary Key | Foreign Keys | Notes |
|---|---|---|---|
| `departments` | `department_id` | — | Parent table; `department_name` is UNIQUE |
| `students` | `student_id` | — | `email` is UNIQUE |
| `instructors` | `instructor_id` | `department_id` → `departments` | Each instructor belongs to one department |
| `courses` | `course_id` | `department_id` → `departments`, `instructor_id` → `instructors` | `fee` CHECK ≥ 0, `level` CHECK constrained |
| `enrollments` | `enrollment_id` | `student_id` → `students`, `course_id` → `courses` | `UNIQUE(student_id, course_id)` prevents duplicate enrollment; `status` defaults to `'Active'` |
| `payments` | `payment_id` | `enrollment_id` → `enrollments` (UNIQUE) | One payment per enrollment; `amount` CHECK ≥ 0 |
| `course_feedback` (bonus) | `feedback_id` | `enrollment_id` → `enrollments` | `rating` CHECK between 1 and 5 |

**Relationship overview:**
- One department → many instructors, many courses
- One instructor → many courses
- One student → many enrollments (many courses, via enrollments)
- One course → many enrollments
- One enrollment → exactly one payment, exactly one feedback record (optional)

The design follows standard normalization principles: each entity (department, student, instructor, course, enrollment, payment, feedback) has its own table, and relationships are captured purely through foreign keys rather than data duplication.

## Closing the Connection

In [82]:
conn.close()
print(f"Connection closed. Database file saved as '{DB_NAME}' in the current directory.")

Connection closed. Database file saved as 'training_academy.db' in the current directory.
